In [18]:
from datasets import load_from_disk, load_dataset, DatasetDict
from collections import Counter
from unsloth import FastModel
import torch
from unsloth.chat_templates import standardize_data_formats, train_on_responses_only
from unsloth.chat_templates import get_chat_template
from transformers import EarlyStoppingCallback
import math
from trl import SFTTrainer, SFTConfig
from transformers.trainer_utils import get_last_checkpoint
import os

torch._dynamo.config.cache_size_limit = 32

SAVE_FOLDER_NAME = "gemma3_4b_it"

In [2]:
dataset = load_from_disk("hf_dataset")
print(dataset)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

train_labels = [example['label'] for example in train_dataset]
label_counts = Counter(train_labels)
print(f"Train Label counts: {label_counts}\n")


test_labels = [example['label'] for example in test_dataset]
label_counts = Counter(test_labels)
print(f"Test Label counts: {label_counts}\n")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 354
    })
})
Train Label counts: Counter({'stress': 1028, 'network': 973})

Test Label counts: Counter({'stress': 182, 'network': 172})



In [3]:
print(f"Sample of dataset: {train_dataset[100]['text']}, {train_dataset[100]['label']}")

Sample of dataset: open5gs-amf logs:

open5gs-ausf logs:

open5gs-bsf logs:

open5gs-nrf logs:

open5gs-nssf logs:

open5gs-pcf logs:

open5gs-scp logs:

open5gs-smf1 logs:

open5gs-smf2 logs:

open5gs-udm logs:
ERROR: Connection timer expired
ERROR: Connection timer expired

open5gs-udr logs:

open5gs-upf1 logs:

open5gs-upf2 logs:

open5gs-webui logs:

ueransim-gnb logs:

ueransim-ue1 logs:

ueransim-ue2 logs:, network


In [4]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

==((====))==  Unsloth 2025.10.3: Fast Gemma3 patching. Transformers: 4.56.2. vLLM: 0.11.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.684 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


In [5]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [6]:
def transform_conversation(sample):
    return {
        'conversations': [
            {
                'from': 'human',
                'value': f"Classify the following 5G fault description. Output only a single word: either 'network' or 'stress'. Do not provide any other text, explanations, or formatting.\n\nFault Description: {sample['text']}"
            },
            {
                 'from': 'gpt',
                'value': sample['label']
            }
        ]
    }

train_dataset = train_dataset.map(transform_conversation)
train_dataset[0]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

{'text': 'open5gs-amf logs:\nINFO: [NSSF] (NRF-notify) NF_DEREGISTERED event\nINFO: (NRF-notify) NF registered\nINFO: [NSSF] (NRF-notify) NF Profile updated\n\nopen5gs-ausf logs:\n\nopen5gs-bsf logs:\n\nopen5gs-nrf logs:\nWARNING: No heartbeat\nINFO: NF de-registered\nWARNING: Not found\nINFO: NF registered [Heartbeat:10s]\nWARNING: NF EndPoint(fqdn) updated [nssf-nnssf:0]\n\nopen5gs-nssf logs:\nERROR: [b37b3c9c-a880-41f0-ba2a-6bac9b7a9c80:NSSF] No heartbeat\nINFO: NF de-registered\nERROR: Connection timer expired\nWARNING: ogs_sbi_client_handler() failed [-3]\nERROR: Connection timer expired\n\nopen5gs-pcf logs:\n\nopen5gs-scp logs:\nINFO: [NSSF] (NRF-notify) NF_DEREGISTERED event\nINFO: RST_STREAM received: stream_id=69\nINFO: RST_STREAM received: stream_id=71\nERROR: on_stream_close_callback() failed (5:STREAM_CLOSED)\nERROR: stream has already been removed\n\nopen5gs-smf1 logs:\n\nopen5gs-smf2 logs:\n\nopen5gs-udm logs:\n\nopen5gs-udr logs:\n\nopen5gs-upf1 logs:\n\nopen5gs-upf2 log

In [7]:
train_dataset = standardize_data_formats(train_dataset)
train_dataset[0]

Unsloth: Standardizing formats (num_proc=16):   0%|          | 0/2001 [00:00<?, ? examples/s]

{'text': 'open5gs-amf logs:\nINFO: [NSSF] (NRF-notify) NF_DEREGISTERED event\nINFO: (NRF-notify) NF registered\nINFO: [NSSF] (NRF-notify) NF Profile updated\n\nopen5gs-ausf logs:\n\nopen5gs-bsf logs:\n\nopen5gs-nrf logs:\nWARNING: No heartbeat\nINFO: NF de-registered\nWARNING: Not found\nINFO: NF registered [Heartbeat:10s]\nWARNING: NF EndPoint(fqdn) updated [nssf-nnssf:0]\n\nopen5gs-nssf logs:\nERROR: [b37b3c9c-a880-41f0-ba2a-6bac9b7a9c80:NSSF] No heartbeat\nINFO: NF de-registered\nERROR: Connection timer expired\nWARNING: ogs_sbi_client_handler() failed [-3]\nERROR: Connection timer expired\n\nopen5gs-pcf logs:\n\nopen5gs-scp logs:\nINFO: [NSSF] (NRF-notify) NF_DEREGISTERED event\nINFO: RST_STREAM received: stream_id=69\nINFO: RST_STREAM received: stream_id=71\nERROR: on_stream_close_callback() failed (5:STREAM_CLOSED)\nERROR: stream has already been removed\n\nopen5gs-smf1 logs:\n\nopen5gs-smf2 logs:\n\nopen5gs-udm logs:\n\nopen5gs-udr logs:\n\nopen5gs-upf1 logs:\n\nopen5gs-upf2 log

In [8]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

def formatting_prompts_func(examples):
   convos = examples["conversations"]
   texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for convo in convos]
   return { "text" : texts, }

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
train_eval_split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_eval_dataset = DatasetDict(
    {
        "train": train_eval_split["train"],
        "validation": train_eval_split["test"],
    }
)
train_dataset = train_eval_dataset["train"]
eval_dataset = train_eval_dataset["validation"]
print(len(train_dataset), len(eval_dataset))

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

1800 201


In [9]:
print(train_dataset[0]['text'])

<start_of_turn>user
Classify the following 5G fault description. Output only a single word: either 'network' or 'stress'. Do not provide any other text, explanations, or formatting.

Fault Description: open5gs-amf logs:
INFO: [PCF] (NRF-notify) NF_DEREGISTERED event
INFO: (NRF-notify) NF registered
INFO: [PCF] (NRF-notify) NF Profile updated

open5gs-ausf logs:

open5gs-bsf logs:

open5gs-nrf logs:
INFO: NF de-registered
INFO: NF registered [Heartbeat:10s]

open5gs-nssf logs:

open5gs-pcf logs:
ERROR: [d93c5466-9f1b-41f0-a9ea-bffa6b08da32:PCF] No heartbeat
INFO: NF de-registered
ERROR: Connection timer expired
ERROR: Connection timer expired

open5gs-scp logs:
INFO: [PCF] (NRF-notify) NF_DEREGISTERED event
INFO: RST_STREAM received: stream_id=289
INFO: RST_STREAM received: stream_id=291
ERROR: on_stream_close_callback() failed (5:STREAM_CLOSED)
INFO: RST_STREAM received: stream_id=293

open5gs-smf1 logs:
INFO: [PCF] (NRF-notify) NF_DEREGISTERED event
INFO: (NRF-notify) NF registered
IN

In [10]:

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3, # Stop after 3 evaluations with no improvement
    early_stopping_threshold=0.01 # A small threshold to prevent stopping on minor fluctuations
)

grad_acc_steps = 4
train_batch_size = 2
steps_per_epoch = len(train_dataset) / (grad_acc_steps * train_batch_size)
print(math.ceil(steps_per_epoch), " is the steps/epoch\n\n")

225  is the steps/epoch




In [12]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[early_stopping_callback],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4, # basically using batch size of 2*4=8
        warmup_steps=22,  # around 10% of steps/epoch. increases lr gradually
        num_train_epochs=3,
        # max_steps=30, # default is -1
        learning_rate=1e-5,
        optim="adamw_8bit",
        weight_decay=0.01, # use low numbers. It penalizes large weights to prevent overfitting
        lr_scheduler_type="cosine",
        seed=3407,
        report_to="tensorboard",
        logging_dir=f"./{SAVE_FOLDER_NAME}/checkpoint/logs",
        logging_steps=22,

        dataset_num_proc=2,
        save_strategy="steps",
        save_steps=22,
        save_total_limit=3,
        greater_is_better=False,
        output_dir=f"./{SAVE_FOLDER_NAME}/checkpoint",
        # evaluation configs
        eval_strategy="steps",
        eval_steps=22,
        per_device_eval_batch_size=2,    # batch size for evaluation
        load_best_model_at_end=True,     # save best model based on eval loss
        metric_for_best_model="eval_loss",

        max_grad_norm=0.5,
        gradient_checkpointing=True, # "unsloth" for reduced memory
    )
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1800 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/201 [00:00<?, ? examples/s]

Masking is to compute the loss, so the model's answer should be masked.

In [13]:
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

Map (num_proc=16):   0%|          | 0/1800 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/201 [00:00<?, ? examples/s]

In [15]:
print(tokenizer.decode(trainer.train_dataset[100]["input_ids"]))

<bos><start_of_turn>user
Classify the following 5G fault description. Output only a single word: either 'network' or 'stress'. Do not provide any other text, explanations, or formatting.

Fault Description: open5gs-amf logs:

open5gs-ausf logs:

open5gs-bsf logs:

open5gs-nrf logs:

open5gs-nssf logs:

open5gs-pcf logs:

open5gs-scp logs:

open5gs-smf1 logs:

open5gs-smf2 logs:
ERROR: Signal-NUM[17] received (Child status change)
ERROR: Signal-NUM[17] received (Child status change)
ERROR: Signal-NUM[17] received (Child status change)
ERROR: Signal-NUM[17] received (Child status change)
ERROR: Signal-NUM[17] received (Child status change)

open5gs-udm logs:

open5gs-udr logs:

open5gs-upf1 logs:

open5gs-upf2 logs:

open5gs-webui logs:

ueransim-gnb logs:

ueransim-ue1 logs:

ueransim-ue2 logs:<end_of_turn>
<start_of_turn>model
stress<end_of_turn>



In [16]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                                                                                                                                            stress<end_of_turn>\n'

In [19]:
if os.path.isdir(trainer.args.output_dir):
    last_checkpoint = get_last_checkpoint(trainer.args.output_dir)
    if last_checkpoint:
        print(f"Resuming training from checkpoint: {last_checkpoint}")
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        trainer.train()
else:
    trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,800 | Num Epochs = 3 | Total steps = 675
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)
/home/sharedrive/nafi/env_unsloth/lib/python3.13/site-packages/torch/_dynamo/guards.py:780: RuntimeWarning: Guards may run slower on Python 3.13.0. Consider upgrading to Python 3.13.1+.
  warnings.warn(
/home/sharedrive/nafi/env_unsloth/lib/python3.13/site-packages/torch/_dynamo/guards.py:780: RuntimeWarning: Guards may run slower on Python 3.13.0. Consider upgrading to Python 3.13.1+.
  warnings.warn(
/home/sharedrive/nafi/env_unsloth/lib/python3.13/site-packages/torch/_dynamo/guards.py:780: RuntimeWarning: Guards may run slower on Python 3.13.0. Consider upgrading to Python 3.13.1+.
  warnings.warn(
/home/sharedrive/nafi/

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
22,8.289100,4.014566
44,1.059400,0.503808
66,0.328300,0.246824
88,0.209000,0.185466
110,0.179600,0.156274
132,0.189500,0.135958
154,0.158800,0.127188
176,0.140600,0.113066
198,0.129500,0.099098
220,0.108700,0.107305


/home/sharedrive/nafi/env_unsloth/lib/python3.13/site-packages/torch/_dynamo/guards.py:780: RuntimeWarning: Guards may run slower on Python 3.13.0. Consider upgrading to Python 3.13.1+.
  warnings.warn(
/home/sharedrive/nafi/env_unsloth/lib/python3.13/site-packages/torch/_dynamo/guards.py:780: RuntimeWarning: Guards may run slower on Python 3.13.0. Consider upgrading to Python 3.13.1+.
  warnings.warn(
Unsloth: Not an error, but Gemma3ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/home/sharedrive/nafi/env_unsloth/lib/python3.13/site-packages/torch/_dynamo/guards.py:780: RuntimeWarning: Guards may run slower on Python 3.13.0. Consider upgrading to Python 3.13.1+.
  warnings.warn(
/home/sharedrive/nafi/env_unsloth/lib/python3.13/site-packages/torch/_dynamo/guards.py:780: RuntimeWarning: Guards may run slower on Python 3.13.0. C

In [20]:
test_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 354
})

In [36]:
correct_count = 0
for sample in test_dataset:
    prompt = f"Classify the following 5G fault description. Output only a single word: either 'network' or 'stress'. Do not provide any other text, explanations, or formatting.\n\nFault Description: {sample['text']}"

    messages = [{
        "role": "user",
        "content": [{"type" : "text", "text" : prompt,}]
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True, # Must add for generation
        tokenize = True,
        return_tensors = "pt",
        return_dict = True,
    )
    outputs = model.generate(
        **inputs.to("cuda"),
        max_new_tokens = 64, # Increase for longer outputs!
        # Recommended Gemma-3 settings!
        temperature = 1.0, top_p = 0.95, top_k = 64,
    )
    text_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    format_resp = text_output[0].split("model")[1]
    print(f"The text prediction: {format_resp.strip()}")
    print(f"The actual output: {sample["label"]}")
    if format_resp.strip() == sample["label"].strip():
        print("CORRECT!!!!!")
        correct_count += 1
    print("\n\n")
print(f"Got {correct_count}/{len(test_dataset)} correct")

The text prediction: network
The actual output: stress



The text prediction: stress
The actual output: stress
CORRECT!!!!!



The text prediction: network
The actual output: network
CORRECT!!!!!



The text prediction: stress
The actual output: stress
CORRECT!!!!!



The text prediction: network
The actual output: network
CORRECT!!!!!



The text prediction: network
The actual output: stress



The text prediction: network
The actual output: network
CORRECT!!!!!



The text prediction: stress
The actual output: stress
CORRECT!!!!!



The text prediction: stress
The actual output: network



The text prediction: network
The actual output: network
CORRECT!!!!!



The text prediction: stress
The actual output: stress
CORRECT!!!!!



The text prediction: stress
The actual output: stress
CORRECT!!!!!



The text prediction: stress
The actual output: stress
CORRECT!!!!!



The text prediction: network
The actual output: network
CORRECT!!!!!



The text prediction: network
The actual output

In [37]:
model.save_pretrained(f"./{SAVE_FOLDER_NAME}/final-save-train")  # Local saving
tokenizer.save_pretrained(f"./{SAVE_FOLDER_NAME}/final-save-train")
# model.save_pretrained_merged(f"./{SAVE_FOLDER_NAME}/final-save-train_merged", tokenizer, save_method = "merged_16bit",)

Found HuggingFace hub cache directory: /home/nafi/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [01:10<00:00, 35.18s/it]


Unsloth: Merge process complete. Saved to `/home/sharedrive/nafi/log_faults/gemma3_4b_it/final-save-train_merged`
